In [1]:
import warnings
warnings.filterwarnings("ignore",category=UserWarning)

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

script_dir = Path().resolve()
env_path = script_dir / ".env"

load_dotenv(dotenv_path=env_path)

print("OPENAPI_KEY:", bool(os.getenv("OPENAPI_KEY")))

os.environ["OPENAPI_KEY"] = os.getenv("OPENAPI_KEY")


OPENAPI_KEY: True


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.5",
                        api_key=os.getenv("OPENAPI_KEY")
)

c:\Users\User\anaconda3\envs\condaenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Define tools

In [4]:
import math
from langchain_core import tools
from langchain_core.tools import tool


@tool
def add(a:float, b:float) -> float:
    """Add two numbers.
    Agent will use this when it detects an addition problem"""
    return a + b

@tool
def multiply(a:float, b:float) -> float:
    """Multiply two numbers.
    Agent will use this when it detects a multiplication problem"""
    return a * b

@tool
def divide(a:float, b:float) -> float:
    """Divide first number by second number.
    Include error handling for division by zero.
    """
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return str(a / b)

@tool
def square_root(a:float) -> float:
    """Square root of a number.
    Error handling for negative numbers."""
    if a < 0:
        raise ValueError("Cannot calculate square root of negative number.")
    return str(math.sqrt(a))


tools = [add, multiply, divide, square_root]

print("===== Available Tools =====")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")
    


===== Available Tools =====
- add: Add two numbers.
Agent will use this when it detects an addition problem
- multiply: Multiply two numbers.
Agent will use this when it detects a multiplication problem
- divide: Divide first number by second number.
Include error handling for division by zero.
- square_root: Square root of a number.
Error handling for negative numbers.


Create the agent -> Loop

In [9]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools
)

Run the agent

In [16]:
def run_agent(question:str):
    """Run the agent and print a clean, beginner-friendly execution trace"""

    print(f"\n User: {question}\n  ")
    print("-" * 60)

    result = agent.invoke({
        "messages":[("user",question)]
    })

    print("Clean Agent Execution Trace:")
    print("-"*60)

    step = 1

    for msg in result["messages"]:

        # 1. Human message
        if msg.type == "human":
            print(f"Step {step}: User asked")
            print(f"  {msg.content}")
            step += 1

        # 2. AI message with tool_calls = agent decides to use a tool
        elif msg.type == "ai" and getattr(msg, "tool_call", None):
            for tool_call in msg.tool_call:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                print(f"{step}.Agent decision")
                print(F" I need to use the tool '{tool_name}' with arguments: {tool_args}")
                step += 1

        #3. Tool message = result retruned by tool
        elif msg.type == "tool":
            print(f"{step}. Tool observation:")
            print(f" Tool returned: {msg.content}")
            step += 1

        #4. Final AI message = final response to user
        elif msg.type == "ai" and msg.content:
            print(f"{step}. Agent final answer: {msg.content}")
            step += 1

    print("-"*60)
    


In [17]:
run_agent("What is 42+58?")


 User: What is 42+58?
  
------------------------------------------------------------
Clean Agent Execution Trace:
------------------------------------------------------------
Step 1: User asked
  What is 42+58?
2. Tool observation:
 Tool returned: 100.0
3. Agent final answer: 100
------------------------------------------------------------


In [18]:
run_agent("what is 15 multipled by 8. Then divide by 3")


 User: what is 15 multipled by 8. Then divide by 3
  
------------------------------------------------------------
Clean Agent Execution Trace:
------------------------------------------------------------
Step 1: User asked
  what is 15 multipled by 8. Then divide by 3
2. Tool observation:
 Tool returned: 120.0
3. Tool observation:
 Tool returned: 40.0
4. Agent final answer: 15 × 8 = 120, and 120 ÷ 3 = **40**.
------------------------------------------------------------


In [19]:
run_agent("I have rectange with length 10 and width 5. What is the area?")


 User: I have rectange with length 10 and width 5. What is the area?
  
------------------------------------------------------------
Clean Agent Execution Trace:
------------------------------------------------------------
Step 1: User asked
  I have rectange with length 10 and width 5. What is the area?
2. Tool observation:
 Tool returned: 50.0
3. Agent final answer: The area of the rectangle is **50 square units**.
------------------------------------------------------------


In [20]:
run_agent("what is 100 devide by 0?")


 User: what is 100 devide by 0?
  
------------------------------------------------------------


ValueError: Cannot divide by zero.